In [23]:
import sys, os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from tqdm.notebook import tqdm

tqdm.pandas()

### Loading package

In [3]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [4]:
from py.utils import verifyDir,verifyFile

In [5]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA21/datasets/', '/media/felipe/DATA21/models/')

### Loading data

In [19]:
DATA_PATH=f"{cfg.DATA_PATH}crimebb/"
SQL_PATH = f"{DATA_PATH}/{cfg.YEAR}/sql/"
CSV_PATH = f"{DATA_PATH}/{cfg.YEAR}/csv/"
CSV_SUMMARY = f"{CSV_PATH}/summary_from_sql/"

In [20]:
verifyDir(CSV_PATH)
verifyDir(CSV_SUMMARY)

In [21]:
year_db = [ db_name for db_name in list_dbs if str(cfg.YEAR) in db_name ]
year_db

['crimebb_2019_deutschland_im_deep_web',
 'crimebb_2019_the_hub',
 'crimebb_2019_garage4hackers',
 'crimebb_2019_kernelmode',
 'crimebb_2019_offensivecommunity',
 'crimebb_2019_raidforums',
 'crimebb_2019_safeskyhacks',
 'crimebb_2019_dread',
 'crimebb_2019_torum',
 'crimebb_2019_greysec',
 'crimebb_2019_mpgh',
 'crimebb_2019_stresserforums',
 'crimebb_2019_envoy_forum',
 'crimebb_2019_antichat',
 'crimebb_2019_hackforums',
 'crimebb_2019_runion']

### Database connection

In [8]:
from py.database import SQLManager

In [9]:
sql_manager = SQLManager()

### Listing tables sizes

In [10]:
list_dbs = sql_manager.listDBs()
list_dbs

Conn closed 1


['crimebb_2019_deutschland_im_deep_web',
 'crimebb_2019_the_hub',
 'crimebb_2019_garage4hackers',
 'crimebb_2019_kernelmode',
 'crimebb_2019_offensivecommunity',
 'crimebb_2019_raidforums',
 'crimebb_2019_safeskyhacks',
 '2021_antichat',
 'crimebb_2019_dread',
 'crimebb_2019_torum',
 'crimebb_2019_greysec',
 'crimebb_2019_mpgh',
 'crimebb_2019_stresserforums',
 'vectordb',
 'crimebb_2019_envoy_forum',
 'crimebb_2019_antichat',
 'crimebb_2019_hackforums',
 '2021_deutschland-im-deep-web',
 '2021_dread',
 '2021_envoy-forum',
 '2021_garage4hackers',
 '2021_raidforums',
 '2021_runion',
 '2021_safe-sky-hacks',
 '2021_the-hub',
 '2021_torum',
 'crimebb_2019_runion']

### List Tables

In [11]:
tables_dict = sql_manager.listDBtables(list_dbs)
tables_dict

{'crimebb_2019_deutschland_im_deep_web': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_the_hub': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_garage4hackers': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_kernelmode': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_offensivecommunity': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_raidforums': ['forum',
  'HF_labelled_posts',
  'HF_labelled_posts_dedup',
  'member',
  'post',
  'reputationvotes',
  'site',
  'thread'],
 'crimebb_2019_safeskyhacks': ['forum',
  

In [12]:
sql_manager.getTableSize(list_dbs)

db: crimebb_2019_deutschland_im_deep_web tables:
                          relname  num_elements     size_mb  size_bytes
0                            post          9983     5896 kB     6037504
1                       Post_pkey          9983      240 kB      245760
2          index_Post_idPost-Site          9983      240 kB      245760
3                          thread          1244      208 kB      212992
4                          member           792      160 kB      163840
5          index_Post_Thread-Site          9983      112 kB      114688
6                Post_Author_Site          9983      104 kB      106496
7          index_Post_Author-Site          9983      104 kB      106496
8      index_Thread_idThread-Site          1244       48 kB       49152
9                     Thread_pkey          1244       48 kB       49152
10                    Member_pkey           792       40 kB       40960
11             Thread_Author_Site          1244       40 kB       40960
12     index_Me

### Converting to CSV

In [13]:
for db_name in tqdm(year_db):
    db_path = f"{CSV_PATH}/{db_name}/"
    verifyDir(db_path)
    sql_manager.cmd_dir_permission(passwd=cfg.USER_PASSWD)
    list_tables = tables_dict[db_name]
    print(list_tables)
    for cur_table in list_tables:
        if "HF" in cur_table:
            continue
        sql_manager.cmd_table_to_csv(f"{db_name}", f"{cur_table}", f"{db_path}/")

  0%|          | 0/16 [00:00<?, ?it/s]

['forum', 'HF_labelled_posts', 'HF_labelled_posts_dedup', 'member', 'post', 'reputationvotes', 'site', 'thread']
Table to CSV ... crimebb_2019_deutschland_im_deep_web-forum
Excecuting command ...  su -l postgres -c "psql -d crimebb_2019_deutschland_im_deep_web -c '\copy forum to /media/felipe/DATA21/datasets/crimebb/2019/csv/crimebb_2019_deutschland_im_deep_web/forum.csv csv header;'"
DB to CSV crimebb_2019_deutschland_im_deep_web FINISHED.

Table to CSV ... crimebb_2019_deutschland_im_deep_web-member
Excecuting command ...  su -l postgres -c "psql -d crimebb_2019_deutschland_im_deep_web -c '\copy member to /media/felipe/DATA21/datasets/crimebb/2019/csv/crimebb_2019_deutschland_im_deep_web/member.csv csv header;'"
DB to CSV crimebb_2019_deutschland_im_deep_web FINISHED.

Table to CSV ... crimebb_2019_deutschland_im_deep_web-post
Excecuting command ...  su -l postgres -c "psql -d crimebb_2019_deutschland_im_deep_web -c '\copy post to /media/felipe/DATA21/datasets/crimebb/2019/csv/crimeb

### Listing CSV files

In [17]:
tables_dict = {}

for db_name in tqdm(year_db):
    db_tables = [db_.split("/")[-1] for db_ in glob.glob(f"{CSV_PATH}/{db_name}/*.csv")]
    tables_dict[db_name] = set(db_tables.copy())
    print(db_name, db_tables)

  0%|          | 0/16 [00:00<?, ?it/s]

crimebb_2019_deutschland_im_deep_web ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_the_hub ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_garage4hackers ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_kernelmode ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_offensivecommunity ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_raidforums ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_safeskyhacks ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_dread ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'forum.csv', 'reputationvotes.csv']
crimebb_2019_torum ['member.csv', 'thread.csv', 'post.csv', 'site.csv', 'foru

### Joining

In [18]:
import itertools
tables_list = list(set(itertools.chain(*tables_dict.values())))
tables_list

['thread.csv',
 'post.csv',
 'member.csv',
 'reputationvotes.csv',
 'forum.csv',
 'site.csv']

In [24]:
for cur_table in tables_list:
    tab_df = pd.DataFrame()
    #for db_name in tqdm(year_db):
    db_path = f"{CSV_PATH}/*/"
    table_db_list = glob.glob(f"{db_path}/{cur_table}")
    print(cur_table, table_db_list)
    tab_df = pd.DataFrame()
    for cur_table_path in tqdm(table_db_list):
        table_df = pd.read_csv(f"{cur_table_path}", sep=',', low_memory=False, on_bad_lines='skip')
        #print(table_df.shape)
        tab_df = pd.concat([tab_df, table_df], ignore_index=True)
    if len(tab_df) > 0:
        tab_df.to_csv(f"{CSV_SUMMARY}{cur_table}", sep='\\', index=False)
        print("num lines:", len(tab_df), tab_df.columns)

thread.csv ['/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_offensivecommunity/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_torum/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_raidforums/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_stresserforums/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_mpgh/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_greysec/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_the_hub/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_antichat/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_envoy_forum/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_garage4hackers/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_kernelmode/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019

  0%|          | 0/16 [00:00<?, ?it/s]

num lines: 5264157 Index(['IdThread', 'Site', 'Author', 'AuthorName', 'Forum', 'Heading',
       'NumPosts', 'LastParse', 'URL', 'parsed', 'NumPages', 'Direction'],
      dtype='str')
post.csv ['/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_offensivecommunity/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_torum/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_raidforums/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_stresserforums/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_mpgh/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_greysec/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_the_hub/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_antichat/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_envoy_forum/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb

  0%|          | 0/16 [00:00<?, ?it/s]

num lines: 54443387 Index(['IdPost', 'Author', 'Thread', 'Timestamp', 'Content', 'AuthorNumPosts',
       'AuthorReputation', 'LastParse', 'parsed', 'Site', 'CitedPost',
       'AuthorName', 'Likes'],
      dtype='str')
member.csv ['/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_offensivecommunity/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_torum/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_raidforums/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_stresserforums/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_mpgh/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_greysec/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_the_hub/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_antichat/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_envoy_forum/member.csv', 

...

num lines: 1857250 Index(['ID', 'Donor', 'Receiver', 'Quantity', 'Reason', 'Timestamp', 'Site'], dtype='str')
forum.csv ['/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_offensivecommunity/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_torum/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_raidforums/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_stresserforums/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_mpgh/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_greysec/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_the_hub/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_antichat/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_envoy_forum/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_garage4hackers/forum.csv', '/media/felipe/DATA21/datasets/

  0%|          | 0/16 [00:00<?, ?it/s]

num lines: 1951 Index(['IdForum', 'NumThreads', 'Site', 'Title', 'LastParse', 'URL', 'parsed',
       'NumPages'],
      dtype='str')
site.csv ['/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_offensivecommunity/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_torum/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_raidforums/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_stresserforums/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_mpgh/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_greysec/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_the_hub/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_antichat/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_envoy_forum/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv/crimebb_2019_garage4hackers/site.csv', '/media/felipe/DAT

  0%|          | 0/16 [00:00<?, ?it/s]

num lines: 16 Index(['IdSite', 'URL', 'Name', 'NumMembers', 'NumForums', 'LastParse',
       'parsed'],
      dtype='str')


In [26]:
if cfg.YEAR in [2018, 2019]:
    import shutil

    csv_files = glob.glob(f"{CSV_SUMMARY}/*.csv")
    print(csv_files)
    for csv_orig in csv_files:
        if "forum" in csv_orig:
            shutil.move(csv_orig, f"{csv_orig.split('forum')[0]}boards.csv")
        elif "votes" in csv_orig:
            shutil.move(csv_orig, f"{csv_orig.split('reputationvotes')[0]}votes.csv")
        else:
            shutil.move(csv_orig, f"{csv_orig.split('.csv')[0]}s.csv")

['/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/member.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/thread.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/post.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/site.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/forum.csv', '/media/felipe/DATA21/datasets/crimebb//2019/csv//summary_from_sql/reputationvotes.csv']
